# Week 8 Solution — Best Value Real Estate

**Domain Swap, Keep Architecture**

A multi-agent RAG pipeline for finding **best-value real estate listings**. Same architecture as "The Price is Right" (Week 8 Day 4–5), applied to the real estate domain.

## Architecture

```
ListingScanner → RAG Estimator (Chroma) → Notify Agent
       ↑                    ↑
   (listings)          (similar comps)
```

## Step 1 — Setup

Imports and configuration. Deal models are in `deal_models/deals.py`.

In [39]:
import os
import sys
import json
import logging
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from sentence_transformers import SentenceTransformer
from deal_models.deals import Deal, DealSelection, Opportunity

load_dotenv(override=True)
openai_client = OpenAI()
MODEL = "gpt-4o-mini"
logging.basicConfig(level=logging.INFO)

## Step 2 — Real Estate Chroma (RAG Corpus)

Sample listings with descriptions and prices for similarity search.

In [40]:
REAL_ESTATE_COMPS = [
    ("3-bed 2-bath single family home, 1,200 sqft, modern kitchen, hardwood floors, suburban neighborhood", 285_000),
    ("2-bed condo downtown, 900 sqft, balcony, gym, doorman", 420_000),
    ("4-bed house, 2,100 sqft, large yard, garage, good schools", 395_000),
    ("1-bed apartment, 650 sqft, walk-up, near transit", 175_000),
    ("5-bed luxury home, 3,500 sqft, pool, chef's kitchen", 1_200_000),
    ("Studio apartment, 450 sqft, renovated, central location", 155_000),
    ("3-bed townhouse, 1,500 sqft, end unit, small patio", 310_000),
    ("2-bed house, 1,000 sqft, fenced yard, quiet street", 245_000),
    ("4-bed waterfront property, 2,400 sqft, dock", 725_000),
    ("2-bed condo, 1,100 sqft, penthouse, city views", 580_000),
]

DB_PATH = "real_estate_vectorstore"

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
summaries = [s[0] for s in REAL_ESTATE_COMPS]
prices = [s[1] for s in REAL_ESTATE_COMPS]
embeddings = encoder.encode(summaries)

client = chromadb.PersistentClient(path=DB_PATH)
try:
    client.delete_collection("listings")
except Exception:
    pass
collection = client.get_or_create_collection("listings", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[str(i) for i in range(len(summaries))],
    embeddings=embeddings.tolist(),
    documents=summaries,
    metadatas=[{"price": p} for p in prices],
)
print(f"Chroma ready: {collection.count()} listings")

## Step 3 — ListingScanner & RAG Estimator

- **ListingScanner**: Mock listings in Deal format.
- **RAG Estimator**: Chroma + OpenAI to estimate fair value.

In [42]:
MOCK_LISTINGS = [
    Deal(product_description="2-bed 1-bath condo, 850 sqft, updated kitchen, near parks and transit.", price=198_000, url="https://example.com/listing1"),
    Deal(product_description="3-bed house, 1,300 sqft, garage, backyard. Needs minor updates.", price=265_000, url="https://example.com/listing2"),
    Deal(product_description="1-bed apartment, 600 sqft, new appliances, downtown.", price=142_000, url="https://example.com/listing3"),
    Deal(product_description="4-bed family home, 2,000 sqft, finished basement, great schools.", price=410_000, url="https://example.com/listing4"),
    Deal(product_description="Studio, 450 sqft, loft style, central.", price=128_000, url="https://example.com/listing5"),
]

def scan_listings() -> str:
    return DealSelection(deals=MOCK_LISTINGS).model_dump_json()

In [43]:
import re

def estimate_fair_value(description: str) -> str:
    vector = encoder.encode([description])
    results = collection.query(query_embeddings=vector.astype(float).tolist(), n_results=5)
    docs = results["documents"][0]
    prices_list = [m["price"] for m in results["metadatas"][0]]
    context = "\n".join([f"Similar: {d[:120]}... | Price: ${p:,.0f}" for d, p in zip(docs, prices_list)])
    prompt = f"Estimate the fair market value (USD) for this property. Use similar listings as reference. Respond with only a number.\n\nProperty: {description}\n\nSimilar listings:\n{context}"
    resp = openai_client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
    raw = resp.choices[0].message.content or "0"
    match = re.search(r"[-+]?\d*\.?\d+", raw.replace(",", ""))
    val = float(match.group()) if match else 0.0
    return f"The estimated fair value of this property is ${val:,.2f}"

In [44]:
def notify_user_of_listing(description: str, list_price: float, estimated_value: float, url: str) -> str:
    gap = estimated_value - list_price
    print(f"Best Value Alert! List=${list_price:,.0f} | Est=${estimated_value:,.0f} | Value gap=${gap:,.0f}")
    print(f"  {description[:80]}...")
    print(f"  {url}")
    return "Notification sent (mock)"

## Step 4 — Planning Agent (Tool Loop)

Same pattern as Day 4: tools and loop until done.

In [45]:
tools = [
    {"type": "function", "function": {"name": "scan_listings", "description": "Returns real estate listings", "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
    {"type": "function", "function": {"name": "estimate_fair_value", "description": "Estimate fair market value from description", "parameters": {"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "notify_user_of_listing", "description": "Notify user of best-value listing; call only once", "parameters": {"type": "object", "properties": {"description": {"type": "string"}, "list_price": {"type": "number"}, "estimated_value": {"type": "number"}, "url": {"type": "string"}}, "required": ["description", "list_price", "estimated_value", "url"], "additionalProperties": False}}},
]

def handle_tool_call(message):
    results = []
    for tc in message.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments or "{}")
        fn = globals().get(name)
        out = fn(**args) if fn else "{}"
        results.append({"role": "tool", "content": out if isinstance(out, str) else json.dumps(out), "tool_call_id": tc.id})
    return results

In [46]:
system_msg = "You find great-value real estate listings. Use your tools to scan, estimate each, pick the best value, and notify once. Then reply OK."
user_msg = """First scan for listings. Estimate fair value for each. Pick the single best (list price much lower than estimate). Notify the user of that one, then reply OK."""
messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]

In [ ]:
done = False
while not done:
    resp = openai_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if resp.choices[0].finish_reason == "tool_calls":
        msg = resp.choices[0].message
        results = handle_tool_call(msg)
        messages.append(msg)
        messages.extend(results)
    else:
        done = True
print("Final reply:", resp.choices[0].message.content)

## Step 5 — Gradio UI

Table: Description, List Price, Estimate, Value Gap, URL. Click row to notify.

In [ ]:
import gradio as gr

def run_pipeline_and_build_table():
    sel = DealSelection.model_validate(json.loads(scan_listings()))
    rows = []
    for d in sel.deals:
        raw = estimate_fair_value(d.product_description)
        m = re.search(r"[-+]?\d*\.?\d+", raw.replace(",", ""))
        est = float(m.group()) if m else 0.0
        gap = max(0, round(est - d.price, 2))
        rows.append([d.product_description, d.price, est, gap, d.url])
    return rows

def on_row_select(rows, evt: gr.SelectData):
    if evt and rows and 0 <= evt.index[0] < len(rows):
        r = rows[evt.index[0]]
        notify_user_of_listing(r[0], r[1], r[2], r[4])

with gr.Blocks(title="Best Value Real Estate", fill_width=True) as ui:
    gr.Markdown('<div style="text-align: center;font-size:24px">Best Value Real Estate</div>')
    gr.Markdown('<div style="text-align: center;font-size:14px">RAG + multi-agent pipeline</div>')
    tbl = gr.Dataframe(headers=["Description", "List Price", "Estimate", "Value Gap", "URL"], wrap=True, column_widths=[4, 1, 1, 1, 2], row_count=10, col_count=5, max_height=400)
    btn = gr.Button("Surface listings")
    state = gr.State([])
    def on_click():
        rows = run_pipeline_and_build_table()
        return rows, rows
    btn.click(fn=on_click, outputs=[state, tbl])
    tbl.select(fn=on_row_select, inputs=[state], outputs=[])

ui.launch(inbrowser=True)